### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Sun Jun 21 12:55:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   74C    P0            250W /  300W |   42179MiB /  81920MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/FaceAgePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/FaceAgeDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0
...,...,...,...
9145,475367_1941-08-03_2011.jpg,70,1.0
9146,304085_1919-07-07_1989.jpg,70,1.0
9147,nm0001627_rm4164078592_1927-2-20_1997.jpg,70,1.0
9148,nm0000024_rm1715129344_1904-4-14_1974.jpg,70,1.0


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)

9150


In [8]:
print(len(all_pc_faceage))

250249


In [9]:
gt_scores = to_numpy(df_faceage['score'].tolist())

In [10]:
max_iter=30000

In [11]:
lr = 0.001

In [12]:
tol = 1e-6

### HTCV

In [13]:
import random
from htcv_m import *

In [14]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
gradem_time = []
for sd in range(10):
    start = time.time()
    grad_em = HTCVWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

 18%|█████████████▋                                                               | 5328/30000 [00:29<02:16, 181.39it/s]



Converged at epoch 5328


 18%|█████████████▋                                                               | 5328/30000 [00:27<02:08, 192.16it/s]



Converged at epoch 5328


 18%|█████████████▋                                                               | 5328/30000 [00:25<01:57, 209.82it/s]


...


Converged at epoch 5328


 18%|█████████████▋                                                               | 5328/30000 [00:29<02:15, 181.85it/s]



Converged at epoch 5328


 18%|█████████████▋                                                               | 5328/30000 [00:27<02:09, 190.40it/s]



Converged at epoch 5328


In [15]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

28.536123299598692 +- 1.5105736328832853


In [16]:
print(f"GradEM -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

GradEM -- Accuracy: 0.791709840297699 ± 0.0, Weighted Accuracy: 0.8732934594154358 ± 0.0, Kendall's Tau: 0.5786494276101927 ± 1.1102230246251565e-16


### HBTL

In [17]:
import random
from grad_em import *

In [18]:
import time
Grad_accs, Grad_waccs, Grad_taus = [], [], []
gradem_time = []
for sd in range(10):
    start = time.time()
    grad_em = GradientEMWrapper(PC_faceage, lr, sd, device, max_iter=max_iter)
    r_est, beta_est = grad_em.run_algorithm()
    end = time.time()
    gradem_time.append(end-start)

    r_est_np = to_numpy(r_est)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
    grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    grad_tau = safe_kendalltau(r_est_np, gt_scores)
    
    Grad_accs.append(grad_acc)    
    Grad_waccs.append(grad_wacc)    
    Grad_taus.append(grad_tau)

 42%|███████████████████████████████▊                                            | 12573/30000 [00:54<01:15, 232.24it/s]


In [19]:
print(f"{np.mean(gradem_time)} +- {np.std(gradem_time)}")

54.20749807357788 +- 2.009142101860675


In [20]:
print(f"GradEM -- Accuracy: {np.mean(Grad_accs)} ± {np.std(Grad_accs)}, Weighted Accuracy: {np.mean(Grad_waccs)} ± {np.std(Grad_waccs)}, Kendall's Tau: {np.mean(Grad_taus)} ± {np.std(Grad_taus)}")

GradEM -- Accuracy: 0.7920328378677368 ± 0.0, Weighted Accuracy: 0.8734515309333801 ± 0.0, Kendall's Tau: 0.5792902305596875 ± 1.1102230246251565e-16


### PG EM

In [21]:
import random
from pgem_initialize_beta_1 import EMWrapper, EMWrapper_3_0

In [22]:
PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda
cuda


  0%|                                                                               | 1/30000 [00:00<2:53:35,  2.88it/s]

Iter 000: Log-likelihood = -0.366355


  0%|▏                                                                             | 77/30000 [00:17<1:53:09,  4.41it/s]

Converged at iter 77, Log-likelihood change = 9.238720e-07
cuda


...

cuda


  0%|                                                                               | 1/30000 [00:00<1:56:12,  4.30it/s]

Iter 000: Log-likelihood = -0.366355


  0%|▏                                                                             | 77/30000 [00:17<1:51:11,  4.49it/s]

Converged at iter 77, Log-likelihood change = 9.238720e-07


In [23]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

18.214092326164245 +- 0.2091115758396223


In [24]:
PGEM_accs

[0.7921259999275208,
 0.7921259999275208,
 0.7921261191368103,
 0.7921260595321655,
 0.7921259999275208,
 0.7921259999275208,
 0.7921259999275208,
 0.7921260595321655,
 0.7921259999275208,
 0.7921259999275208]

In [25]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7921260237693787 ± 3.95372484964776e-08, Weighted Accuracy: 0.8736114144325257 ± 2.9200193199910855e-08, Kendall's Tau: 0.5794749917088768 ± 6.321289750754285e-08


### Faster PGEM

In [26]:
import time

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
f_pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper_3_0(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    print("Elapsed:", end - start, "seconds")
    f_pgem_time.append(end-start)
    
    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  0%|                                                                                 | 4/30000 [00:00<15:09, 32.97it/s]

Iter 000: Log-likelihood = -0.367956


  0%|▏                                                                               | 62/30000 [00:01<13:48, 36.12it/s]


Converged at iter 62, ΔLL = 9.834766e-07
Elapsed: 2.589231252670288 seconds
cuda


...

Converged at iter 62, ΔLL = 9.834766e-07
Elapsed: 2.3746449947357178 seconds
cuda


  0%|                                                                                 | 8/30000 [00:00<12:54, 38.71it/s]

Iter 000: Log-likelihood = -0.367956


  0%|▏                                                                               | 63/30000 [00:01<13:52, 35.97it/s]

Converged at iter 63, ΔLL = 9.238720e-07
Elapsed: 2.811068534851074 seconds


In [27]:
print(f"{np.mean(f_pgem_time)} +- {np.std(f_pgem_time)}")

2.623660683631897 +- 0.1862289951607676


In [28]:
PGEM_accs

[0.7920684814453125,
 0.792068362236023,
 0.7920685410499573,
 0.7920684814453125,
 0.7920684814453125,
 0.792068362236023,
 0.792068362236023,
 0.7920684814453125,
 0.7920684814453125,
 0.7920683026313782]

In [29]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7920684337615966 ± 7.44461774635124e-08, Weighted Accuracy: 0.8735842823982238 ± 8.760057959973256e-08, Kendall's Tau: 0.5793607636196549 ± 1.5889510695642438e-07


### BT

In [30]:
BT_accs, BT_waccs, BT_taus, BT_times = [], [], [], []

for i in range(10):
    start = time.time()
    bt_scores = opt_fair.opt_pairwise_gpu(size, all_pc_faceage, alpha=0, initial_params=None, max_iter=max_iter, tol=tol)
    end = time.time()
    BT_times.append(end-start)
    
    r_est_np = to_numpy(bt_scores)
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    bt_acc = compute_acc(df_faceage, 1*r_est_np, device)
    bt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    bt_tau = safe_kendalltau(r_est_np, gt_scores)
    
    BT_accs.append(bt_acc)
    BT_waccs.append(bt_wacc)    
    BT_taus.append(bt_tau)    

In [31]:
print(f"{np.mean(BT_times)} +- {np.std(BT_times)}")

3.8780869960784914 +- 0.39398787914662653


In [32]:
print(f"Simple BT -- Accuracy: {np.mean(BT_accs)} ± {np.std(BT_accs)}, Weighted Accuracy: {np.mean(BT_waccs)} ± {np.std(BT_waccs)}, Kendall's Tau: {np.mean(BT_taus)} ± {np.std(BT_taus)}")

Simple BT -- Accuracy: 0.7900682687759399 ± 0.0, Weighted Accuracy: 0.8719562292098999 ± 0.0, Kendall's Tau: 0.5753930665630511 ± 0.0


### BARP

In [33]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()
classes = df_faceage['gender']
BARP_times = []

BARP_accs = []
BARP_waccs = []
BARP_taus = []


for i in range(10):
    start = time.time()
    FaceAge = opt_fair.BARP(data = PC_faceage, penalty = 0, classes = classes, device=device)
    annot_bt_temp, annot_bias =  opt_fair._alternate_optim_torch(size, num_reviewers, FaceAge, lr_x=lr, lr_y=lr, iters = max_iter)
    end = time.time()
    BARP_times.append(end-start)
    
    r_est_np = to_numpy(annot_bt_temp)
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    barp_acc = compute_acc(df_faceage, 1*r_est_np, device)
    barp_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    barp_tau = safe_kendalltau(r_est_np, gt_scores)
    
    BARP_accs.append(barp_acc)
    BARP_waccs.append(barp_wacc)   
    BARP_taus.append(barp_tau)    

 34%|█████████████████████████▌                                                  | 10089/30000 [00:31<01:01, 325.33it/s]


In [34]:
print(f"{np.mean(BARP_times)} +- {np.std(BARP_times)}")

31.394195890426637 +- 2.07163335816158


In [35]:
print(f"BARP -- Accuracy: {np.mean(BARP_accs)} ± {np.std(BARP_accs)}, Weighted Accuracy: {np.mean(BARP_waccs)} ± {np.std(BARP_waccs)}, Kendall's Tau: {np.mean(BARP_taus)} ± {np.std(BARP_taus)}")

BARP -- Accuracy: 0.7908041477203369 ± 0.0, Weighted Accuracy: 0.8726015090942383 ± 0.0, Kendall's Tau: 0.5768528788223168 ± 0.0


### RC

In [36]:
RC_times, RC_accs, RC_waccs, RC_taus = [], [], [], []

for i in range(10):
    start = time.time()
    rc_obj = RankCentrality(device)
    A = rc_obj.matrix_of_comparisons(size, all_pc_faceage)
    # print("A matrix done")
    P = rc_obj.trans_prob(A)
    # print("P matrix done")
    pi = rc_obj.stationary_dist(P, max_iter=max_iter, tol=tol)
    end = time.time()
    RC_times.append(end-start)
    rank_centrality_scores = torch.log(pi).cpu().numpy()
    
    r_est_np = to_numpy(rank_centrality_scores)
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    rc_acc = compute_acc(df_faceage, 1*r_est_np, device)
    rc_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    rc_tau = safe_kendalltau(r_est_np, gt_scores)
    
    RC_accs.append(rc_acc)
    RC_waccs.append(rc_wacc)    
    RC_taus.append(rc_tau)    

In [37]:
print(f"RC -- Accuracy: {np.mean(RC_accs)} ± {np.std(RC_accs)}, Weighted Accuracy: {np.mean(RC_waccs)} ± {np.std(RC_waccs)}, Kendall's Tau: {np.mean(RC_taus)} ± {np.std(RC_taus)}")

RC -- Accuracy: 0.7804015278816223 ± 0.0, Weighted Accuracy: 0.8644048571586609 ± 0.0, Kendall's Tau: 0.5582073434385082 ± 0.0


In [38]:
print(f"{np.mean(RC_times)} +- {np.std(RC_times)}")

0.07546451091766357 +- 0.01673748085225677


### CrowdBT

In [39]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [40]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

cuda


In [ ]:
CrowdBT_accs, CrowdBT_waccs, CrowdBT_taus = [], [], []
K = num_reviewers
gt_df = df_faceage
crowdbt_time = []
for seed in range(10):
    try:
        start = time.time()
        crowdbt_test = opt_fair.CrowdBT_3_0(data=PC_faceage, penalty=0, device=device, random_seed=seed)
        crowdbt_scores, _ = crowdbt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, iters=max_iter)
        end = time.time()
        crowdbt_time.append(end-start)
        r_est_np = to_numpy(crowdbt_scores)
        gt_scores = to_numpy(df_faceage['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        crowdbt_acc = compute_acc(df_faceage, 1*r_est_np, device)
        crowdbt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
        crowdbt_tau = safe_kendalltau(r_est_np, gt_scores)
        CrowdBT_accs.append(crowdbt_acc)
        CrowdBT_waccs.append(crowdbt_wacc)
        CrowdBT_taus.append(crowdbt_tau)
    except Exception as e:
        print(e)
        CrowdBT_accs.append(0.0)
        CrowdBT_waccs.append(0.0)
        CrowdBT_taus.append(0.0)

 15%|█████████▌                                                     | 4558/30000 [00:24<02:56, 144.00it/s, loss=7.23e+4]

In [53]:
print(f"{np.mean(crowdbt_time)} +- {np.std(crowdbt_time)}")

122.67742388248443 +- 2.6677594375967097


In [54]:
print(f"CrowdBT -- Accuracy: {np.mean(CrowdBT_accs)} ± {np.std(CrowdBT_accs)}, Weighted Accuracy: {np.mean(CrowdBT_waccs)} ± {np.std(CrowdBT_waccs)}, Kendall's Tau: {np.mean(CrowdBT_taus)} ± {np.std(CrowdBT_taus)}")

CrowdBT -- Accuracy: 0.7917298078536987 ± 0.0, Weighted Accuracy: 0.8730530738830566 ± 0.0, Kendall's Tau: 0.5786890421159417 ± 0.0


### FactorBT

In [ ]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [ ]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

In [ ]:
from collections import defaultdict
from pathlib import Path

def build_pc_dict_for_noisybt(df, df_faceage, worker_col="worker", left_col="left", right_col="right", label_col="label"):
    """
    Builds the dict NoisyBT_3_0 expects: {worker_id: [(left, right, winner), ...]}

    Mirrors the exact item/worker id mapping used to build PC_faceage_spm:
      - worker ids: position of first appearance in df[worker_col].unique()
      - item ids: position of df_faceage['full_path'] (matched via Path(...).name,
        since df['left']/df['right']/df['label'] are full URLs but full_path
        entries are basenames)

    Unlike PC_faceage_spm (which collapses each row to (winner, loser)), this
    keeps the left/right slot assignment, since NoisyBT's bias parameter needs
    to know which item was shown in the "left" position.

    Returns the dict AND the two id mappings, so you can invert them later
    if needed (e.g. to map scores back to filenames).
    """
    unique_performers = list(df[worker_col].unique())
    performer_label_dict = {performer: i for i, performer in enumerate(unique_performers)}

    item_labels = list(df_faceage["full_path"])
    item_label_dict = {item: i for i, item in enumerate(item_labels)}

    pc_dict = defaultdict(list)
    for performer, group in df.groupby(worker_col):
        key = performer_label_dict[performer]
        for _, row in group.iterrows():
            left = Path(row[left_col]).name
            right = Path(row[right_col]).name
            winner = Path(row[label_col]).name

            left_id = item_label_dict[left]
            right_id = item_label_dict[right]
            winner_id = item_label_dict[winner]

            pc_dict[key].append((left_id, right_id, winner_id))

    return dict(pc_dict), performer_label_dict, item_label_dict

In [ ]:
def sort_df(df, column_name):
    # Sort by a specific column (replace 'column_name' with your column)
    df_sorted = df.sort_values(by=column_name, ascending=True)  # or ascending=False

    return df_sorted

df = pd.read_csv('data/crowd_labels.csv')
df = df.rename(columns={'performer': 'worker'})

In [ ]:
df.head()

In [ ]:
gt_df = df_faceage
gt_df.head()

In [ ]:
import time
NoisyBT_accs, NoisyBT_waccs, NoisyBT_taus = [], [], []
K = num_reviewers
gt_df = df_faceage
noisybt_time = []

# Build the (left, right, winner) dict NoisyBT_3_0 needs, from the same
# crowd_labels.csv-derived df used by the original NoisyBradleyTerry.
# `df` must have integer-coded 'worker', 'left', 'right', 'label' columns
# aligned with `size`/num_items and gt_df['score'] (see build_pc_dict.py).
PC_faceage_noisybt, _performer_label_dict, _item_label_dict = build_pc_dict_for_noisybt(df, df_faceage)

for seed in range(10):
    try:
        start = time.time()
        noisybt_test = opt_fair.NoisyBT_3_0(data=PC_faceage_noisybt, penalty=0, device=device, random_seed=seed)
        noisybt_scores, noisybt_skills, noisybt_biases = noisybt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, iters=max_iter)
        end = time.time()
        noisybt_time.append(end-start)
        r_est_np = to_numpy(noisybt_scores)
        gt_scores = to_numpy(gt_df['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        noisybt_acc = compute_acc(gt_df, 1*r_est_np, device)
        noisybt_wacc = compute_weighted_acc(gt_df, 1*r_est_np, device)
        noisybt_tau = safe_kendalltau(r_est_np, gt_scores)
        NoisyBT_accs.append(noisybt_acc)
        NoisyBT_waccs.append(noisybt_wacc)
        NoisyBT_taus.append(noisybt_tau)
    except Exception as e:
        print(e)
        NoisyBT_accs.append(0.0)
        NoisyBT_waccs.append(0.0)
        NoisyBT_taus.append(0.0)

In [55]:
print(f"FactorBT -- Accuracy: {np.mean(NoisyBT_accs)} ± {np.std(NoisyBT_accs)}, Weighted Accuracy: {np.mean(NoisyBT_waccs)} ± {np.std(NoisyBT_waccs)}, Kendall's Tau: {np.mean(NoisyBT_taus)} ± {np.std(NoisyBT_taus)}")

FactorBT -- Accuracy: 0.7916303873062134 ± 0.0, Weighted Accuracy: 0.8729999661445618 ± 0.0, Kendall's Tau: 0.5784917475293071 ± 1.1102230246251565e-16


In [56]:
print(f"{np.mean(noisybt_time)} +- {np.std(noisybt_time)}")

147.87738058567047 +- 3.0701617783266064
